In [ ]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("./ARK_INNOVATION_ETF_ARKK_HOLDINGS.pdf")
print(loader)

In [ ]:
docs = loader.load()
print(len(docs))

In [ ]:
print(docs[0].metadata)

In [ ]:
print(docs[0].page_content[:100])

In [ ]:
print(docs[0].page_content[:500])

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("./ARK_INNOVATION_ETF_ARKK_HOLDINGS.pdf")

docs = loader.load()

print(docs[0].metadata)

In [ ]:
print(docs[0].page_content[:100])

In [ ]:
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

file_path = "./ARK_INNOVATION_ETF_ARKK_HOLDINGS.pdf"
loader = PDFPlumberLoader(file_path)

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)

recursive_docs = text_splitter.split_documents(docs)

In [ ]:
# print(recursive_docs)

In [ ]:
print(len(recursive_docs))

In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

load_dotenv()

API_KEY = os.getenv("NVIDIA")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")


embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

text = "랭체인이 뭔가요?"

vectorstore = InMemoryVectorStore.from_texts([text], embedding=embeddings)

print(vectorstore)

In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_EMBEDDING")

docs = [
    "고양이가 물을 마신다",
    "강아지가 물을 마신다",
    "사람이 커피를 마신다",
    "눈에 보이지 않는 것의 엄청난 힘",
    "메리 크리스마스",
]

metadatas = [
    {"id": 1, "topic": "동물"},
    {"id": 2, "topic": "동물"},
    {"id": 3, "topic": "사람"},
    {"id": 4, "topic": "문구"},
    {"id": 5, "topic": "인사"},
]

emb = NVIDIAEmbeddings(model=MODEL, api_key=API_KEY)

db = FAISS.from_texts(docs, emb, metadatas=metadatas)

query = "고양이가 음료를 마신다"

result = db.similarity_search(query, k=5)

for r in result:
    print(r.page_content)

In [ ]:
scored_result = db.similarity_search_with_score(query, k=5)

for doc, score in scored_result:
    print(f"문장: {doc.page_content}")
    print(f"거리(score): {score:.4f}")
    print("-" * 40)

In [ ]:
index = db.index
print("index.ntotal : ", index.ntotal)
print()
print("index.d : ", index.d)

vector = index.reconstruct(0)
print()
print("vector : ", vector[:10])

print()
print("doc ID list : ", db.docstore._dict.keys())

first_id = list(db.docstore._dict.keys())[0]
print()
print("first doc : ", first_id)

In [ ]:
print("저장된 모든 문서와 메타데이터")
print()
for idx, doc in enumerate(db.docstore._dict.values()):
    vector = db.index.reconstruct(idx)

    print("문서 내용 : ", doc.page_content)
    print("메타 데이터 : ", doc.metadata)
    print("벡터 차원 : ", vector.shape)
    print("벡터 상위 5개 값 : ", vector[:7])
    print()

In [ ]:
DB_PATH = "./faiss_test"
db.save_local(DB_PATH)

In [ ]:
new_db = FAISS.load_local(
    folder_path=DB_PATH, embeddings=emb, allow_dangerous_deserialization=True
)

query = "고양이는 무엇을 마시나요?"

docs = new_db.similarity_search(query)
print(docs)

In [3]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pdf_utils import clean_pdf_documents

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_EMBEDDING")

PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

DB_PATH = "./faiss_index_kofia_guide_cleaned"

embeddings = NVIDIAEmbeddings(model=MODEL, api_key=API_KEY)

if os.path.exists(DB_PATH):
    vectorstore = FAISS.load_local(
        folder_path=DB_PATH, embeddings=embeddings, allow_dangerous_deserialization=True
    )
else:
    loader = PyMuPDFLoader(PDF_PATH)
    docs = loader.load()

    docs = clean_pdf_documents(docs)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500, chunk_overlap=100, separators=["\n\n", ".", " ", ""]
    )

    split_docs = text_splitter.split_documents(docs)

    for idx, d in enumerate(split_docs):
        d.metadata["id"] = idx

    print("원본 페이지 수 : ", len(docs))
    print("분할된 청크 수 : ", len(split_docs))

    vectorstore = FAISS.from_documents(split_docs, embeddings)
    vectorstore.save_local(DB_PATH)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

python-dotenv could not parse statement starting at line 30


/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


docs :  [Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Adobe InDesign CS4 (6.0)', 'creationdate': '2018-12-27T10:04:51+09:00', 'source': './rag_data/금융투자협회_투자길라잡이_2018.pdf', 'file_path': './rag_data/금융투자협회_투자길라잡이_2018.pdf', 'total_pages': 191, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2018-12-28T11:19:21+09:00', 'trapped': '', 'modDate': "D:20181228111921+09'00'", 'creationDate': "D:20181227100451+09'00'", 'page': 0}, page_content='투자자의 눈높이에 맞게 다양한 금융상품별 투자요령, 계좌관리요령 등에 대한 설명,\n그리고 판례와 법리, 분쟁조정사례 등을 통해 불완전판매나 금융사고에 의한\n손해를 예방하고 해결하는 방법을 제시한 투자자용 투자문화 지침서\n금융투자회사 임직원에게는 분쟁예방 역량강화와 투자자 보호를 위한 활용서'), Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Adobe InDesign CS4 (6.0)', 'creationdate': '2018-12-27T10:04:51+09:00', 'source': './rag_data/금융투자협회_투자길라잡이_2018.pdf', 'file_path': './rag_data/금융투자협회_투자길라잡이_2018.pdf', 'total_pages': 191, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'key

In [4]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, temperature=0.1)


def format_docs(docs):
    parts = []
    for d in docs:
        page = d.metadata.get("page", "N/A")
        parts.append(f"[page {page}\n{d.page_content}]")
    return "\n\n".join(parts)


base_system_instruction = (
    "너는 금융 질문에 답하는 AI 어시스턴트야. "
    "답은 가능하면 핵심 명사구나 짧은 한 줄로만 하고, 불릿이나 긴 설명은 하지 말 것."
)


prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            base_system_instruction
            + " 이번에는 반드시 제공된 문서 컨텍스트 안의 내용만 근거로 답해. "
            + "컨텍스트에 정답이 없으면 '제공된 문서 내용에서는 찾을 수 없습니다'라고 답해.",
        ),
        (
            "user",
            (
                "질문: {question}\n\n"
                "참고할 문서 컨텍스트:\n"
                "--------------------\n"
                "{context}\n"
                "--------------------\n"
                "위 문서 내용만 근거로 답해줘."
            ),
        ),
    ]
)

no_rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            base_system_instruction
            + " 이번에는 참고할 문서 컨텍스트가 없다고 가정하고, 네 일반 지식만으로 답해. "
            + "모르면 억지로 지어내지 말고 잘 모르겠다고 답해.",
        ),
        (
            "user",
            (
                "질문: {question}\n\n"
                "참고할 문서 컨텍스트는 제공되지 않습니다.\n"
                "네 일반 지식만으로 답해줘."
            ),
        ),
    ]
)


rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever | RunnableLambda(format_docs),
    }
    | prompt
    | llm
    | StrOutputParser()
)

no_rag_chain = no_rag_prompt | llm | StrOutputParser()


question = "피싱사기로 개인정보를 알려줬을 때 가까운 은행에 등록 요청하라고 한 시스템 이름은 무엇인가요?"

rag_answer = rag_chain.invoke(question)
print("rag_answer : ", rag_answer)
print()
no_rag_answer = no_rag_chain.invoke(question)
print("no_rag_answer :", no_rag_answer)

rag_answer :  개인정보 노출자 사고예방 시스템



ReadTimeout: HTTPSConnectionPool(host='integrate.api.nvidia.com', port=443): Read timed out. (read timeout=60)

In [5]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

loader = PyPDFLoader(PDF_PATH)

docs = clean_pdf_documents(loader.load())

docs[58].page_content[:320]

'2. 일임매매\n일임매매란?\n일임매매 관련 기본 법리\n자본시장법상 일임매매(투자일임)는 금융위원회에 투\n자일임업자로 등록된 금융투자업자가 투자자로부터 금\n융투자상품에 대한 투자판단의 전부 또는 일부를 일임\n받아 금융투자상품을 취득·처분, 그 밖의 방법으로 운\n용하는 것을 의미합니다.\n종전 증권거래법의 경우, 위탁매매를 수행하는 증권회\n사 또는 선물회사에 대해 제한적인 범위 내에서 일임\n매매를 허용하는 명문규정을 두었으나 현행 자본시장\n법에서는 이러한 조항이 별도로 규정되어 있지는 않\n습니다.\n다만, 현행 자본시장법에서도 투자일임업자로 등록된 금융투자회사가 고객과 투자일'

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

full_text = "\n".join(doc.page_content for doc in docs)

fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=100, length_function=len
)

fixed_chunks = fixed_splitter.split_text(full_text)

print("고정 길이 청크 개수:", len(fixed_chunks))
print("고정 길이 청크 예시(둘째 청크):")
print(fixed_chunks[1][:300].replace("\n", " "))

고정 길이 청크 개수: 507
고정 길이 청크 예시(둘째 청크):
선택이 아닌 필수 조건이 되었습니다. 금융투자업계는 이러한 시대적 요구에 맞게 금융소비자가 상품을 충분히 이해하고 자신에게 가장 적합한 상품을 선택할 수 있도록 금융소비자를 지원하는 금융소비자보호 조직을 강화하는 한편, 불완전판매 예방을 위하여 임직원 교육을 꾸준히 실시해 오고 있으며, 금융소비자의 자기 보호능력 향상을 위해 다양한 투자교육 프로그램을 지원하는 등 최선의 노력을 다하고 있습니다. 금융투자협회는 이러한 금융소비자보호 노력의 일환으로 최근 주요 이슈를 반영하여 기존의 분쟁조정사례·판례 핸드북을 보강한 증보판을 발간하였습
